# Script to parse jedi files

In [ ]:
import glob
import numpy as np
import pandas as pd
import datetime as dt

import matplotlib.pyplot as plt 
import matplotlib.colors as colors
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

from JunoPy.jedi import JediChannel, JediData
from JunoPy.index import IndexFile

In [ ]:
startevent = dt.datetime.strptime('2017-236T20:21:15', '%Y-%jT%H:%M:%S')
endevent   = dt.datetime.strptime('2017-236T20:21:38', '%Y-%jT%H:%M:%S')
start = startevent - dt.timedelta(minutes=30)
end   = endevent + dt.timedelta(minutes=30)

jedifiles = glob.glob('data/jedi/*SISP_CDR_{:s}_*.TAB'.format(start.strftime('%Y%j')))
if not jedifiles:
    jediindex = IndexFile(dataset='jedi')
    jediindex.get_files_between_dates(
        start, end, DoDownload=True, sid='ERSISP')
    
jedidatas = []
for jedifile in jedifiles:
    jedidatas.append(JediData(filename=jedifile))
    print(jedifile)
    
jedidata = jedidatas[0]
jedidata.df = jedidata.df.loc[start:end]

In [ ]:
jedidata.df.head()

##  Create the spectrogram data we need
X - time  
Y - frequency bins  
C - flux added from each look direction  

In [ ]:


df = jedidata.df
df['endtime'] = df.apply(lambda x: x.name + dt.timedelta(seconds=x['Duration']), axis=1)

x = np.zeros(len(df) + 1, dtype=object)
x[:-1] = df.index
x[-1] = df['endtime'][-1]
    
fig, ax = plt.subplots(6, 1, sharex=True, figsize=(11, 8.5), gridspec_kw=dict(hspace=0))

for iax, a in enumerate(ax):

    columns = [x for x in jedidata.FluxChannels.columns if 'T{:d}'.format(iax) in x]
    freq_lower = jedidata.FluxChannels.lowE[iax * len(columns): (iax + 1) * len(columns)]
    freq_upper = jedidata.FluxChannels.highE[iax * len(columns): (iax + 1) * len(columns)]
    
    y = np.zeros(len(freq_lower) * 2)
    j = 0
    for k in range(len(freq_lower)):
        y[j] = float(freq_lower[k])
        y[j + 1] = float(freq_upper[k])
        j += 2

    c0 = df[columns].values.T
    c = np.zeros((y.shape[0] - 1, x.shape[0] - 1))
    for i in range(c.shape[0]):
        for j in range(c.shape[1]):
            if i % 2 != 0:
                c[i, j] = np.NaN
            else:
                c[i, j] = c0[int(i/2), j]
    print(c.shape)


    mp = a.pcolormesh(x, y, c, cmap='jet', norm=colors.LogNorm(), vmin=100, vmax=10**5)
    a.set_yscale('log')
    a.set_ylim([20, y[-1]])

    size = 0.02
    bbox = a.get_position()
    cax = a.figure.add_axes([bbox.x1, bbox.y0, 0.02, bbox.y1- bbox.y0])
    cbar = a.figure.colorbar(mp, cax=cax)
    a.label_outer()
        
plt.show()

In [ ]:
y

## 3. Plot the pitch angle spectrogram. 

In [ ]:

# Calculate the pitch angle - 
# Integrate the energy flux in the channel. 

df = jedidata.df
df['endtime'] = df.apply(lambda x: x.name + dt.timedelta(seconds=x['Duration']), axis=1)

def calc_broadband_flux(x, lowE, upE):
    sum = 0
    for i, val in enumerate(x.values):
        if val > 0:
            sum += val * (float(upE[i]) - float(lowE[i]))
    return sum
    
for iax in range(6):
    columns = [x for x in jedidata.FluxChannels.columns if 'T{:d}'.format(iax) in x]
    freq_lower = jedidata.FluxChannels.lowE[iax * len(columns): (iax + 1) * len(columns)]
    freq_upper = jedidata.FluxChannels.highE[iax * len(columns): (iax + 1) * len(columns)]

    df['BroadbandFlux_T{:d}'.format(iax)] = df.apply(
        lambda x: calc_broadband_flux(
            x[columns], freq_lower, freq_upper), axis=1)


        

In [ ]:
# for iax in range(6):
    # df['BroadbandFlux_T{:d}'.format(iax)].plot(logy=True, figsize=(11, 8.5), legend=True, lw=1.0, ls='dotted')
df.plot(y=['BroadbandFlux_T{:d}'.format(iax) for iax in range(6)], figsize=(11,8.5), legend=True, markersize=11., style='.')

In [ ]:
df.plot(y=['Pitch_Angle_T{:d}'.format(iax) for iax in range(6)], figsize=(11,8.5), legend=True, markersize=11., style='.')

## Multiple JEDI files (090-180-270) and making the pitch angle versus time spectra. 

In [ ]:
start = dt.datetime.strptime('2019-089T20:35:40', '%Y-%jT%H:%M:%S') - dt.timedelta(minutes=5)
end   = dt.datetime.strptime('2019-089T20:36:36', '%Y-%jT%H:%M:%S') + dt.timedelta(minutes=5)

jedifiles = glob.glob('data/jedi/*_{:s}_*.TAB'.format(start.strftime('%Y%j')))

